# Clean and aggregate all months of 2025

Follow the 2024 cleaning rules: remove pickup zones 264 and 265, retain pickups within the file's calendar month, and count trips per hour and zone. Save each cleaned month and each hourly month, then combine the hourly data for the full year. Hours with no recorded pickups are filled with zero for zones 1–263.

The helpers below use 2025 explicitly; the existing 2024 helpers in `data_tools.py` are unchanged.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

from pathlib import Path
import pandas as pd
import data_tools as data

year = 2025
raw_dir = Path.home() / "Downloads"
project_dir = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
data_dir = project_dir / "data"
data_dir.mkdir(exist_ok = True)

input_files = {month: raw_dir / f"yellow_tripdata_{year}-{month:02d}.parquet" for month in range(1, 13)}
missing_files = [str(path) for path in input_files.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Missing monthly files: " + ", ".join(missing_files))
monthly_summary = {}


In [ ]:
def process_month_2025(month):
    raw = pd.read_parquet(input_files[month])
    start = pd.Timestamp(year = year, month = month, day = 1)
    end = start + pd.DateOffset(months = 1)

    cleaned = data.clean(raw).copy()
    cleaned["tpep_pickup_datetime"] = pd.to_datetime(cleaned["tpep_pickup_datetime"])
    cleaned = cleaned.loc[
        (cleaned["tpep_pickup_datetime"] >= start)
        & (cleaned["tpep_pickup_datetime"] < end)
    ].copy()
    assert cleaned["PULocationID"].isin(range(1, 264)).all()
    cleaned_path = data_dir / f"yellow_taxi_{year}_{month:02d}_cleaned.parquet"
    cleaned.to_parquet(cleaned_path, index = False)

    hours = pd.date_range(start = start, end = end, freq = "h", inclusive = "left")
    full_index = pd.MultiIndex.from_product(
        [hours, range(1, 264)], names = ["tpep_pickup_datetime", "PULocationID"]
    )
    hourly = (
        cleaned.groupby([pd.Grouper(key = "tpep_pickup_datetime", freq = "h"), "PULocationID"])
        .size()
        .reindex(full_index, fill_value = 0)
        .rename("trip_count")
        .reset_index()
    )
    assert len(hourly) == len(hours) * 263
    assert hourly["trip_count"].sum() == len(cleaned)
    hourly_path = data_dir / f"yellow_taxi_{year}_{month:02d}_aggreg.parquet"
    hourly.to_parquet(hourly_path, index = False)
    monthly_summary[month] = {
        "month": month, "raw_trips": len(raw), "cleaned_trips": len(cleaned),
        "removed_trips": len(raw) - len(cleaned), "zone_hours": len(hourly),
    }
    print(f"{year}-{month:02d}: {len(cleaned):,} cleaned pickups; {len(hourly):,} zone-hours. Saved cleaned and hourly files.")
    return hourly


### January


In [ ]:
df_jan = process_month_2025(1)


### February


In [ ]:
df_feb = process_month_2025(2)


### March


In [ ]:
df_mar = process_month_2025(3)


### April


In [ ]:
df_apr = process_month_2025(4)


### May


In [ ]:
df_may = process_month_2025(5)


### June


In [ ]:
df_jun = process_month_2025(6)


### July


In [ ]:
df_jul = process_month_2025(7)


### August


In [ ]:
df_aug = process_month_2025(8)


### September


In [ ]:
df_sep = process_month_2025(9)


### October


In [ ]:
df_oct = process_month_2025(10)


### November


In [ ]:
df_nov = process_month_2025(11)


### December


In [ ]:
df_dec = process_month_2025(12)


### Combine and validate the full year


In [ ]:
df_2025 = pd.concat([df_jan, df_feb, df_mar, df_apr, df_may, df_jun, df_jul, df_aug, df_sep, df_oct, df_nov, df_dec], ignore_index = True)
df_2025 = df_2025.sort_values(["tpep_pickup_datetime", "PULocationID"]).reset_index(drop = True)

expected_hours = pd.date_range("2025-01-01", "2026-01-01", freq = "h", inclusive = "left")
expected_index = pd.MultiIndex.from_product(
    [expected_hours, range(1, 264)], names = ["tpep_pickup_datetime", "PULocationID"]
)
actual_index = pd.MultiIndex.from_frame(df_2025[["tpep_pickup_datetime", "PULocationID"]])
assert actual_index.equals(expected_index)
assert len(df_2025) == 365 * 24 * 263
assert df_2025["trip_count"].notna().all()
assert (df_2025["trip_count"] >= 0).all()
assert df_2025["trip_count"].sum() == sum(row["cleaned_trips"] for row in monthly_summary.values())

output_path = data_dir / "yellow_taxi_2025_aggreg.parquet"
df_2025.to_parquet(output_path, index = False)
pd.testing.assert_frame_equal(pd.read_parquet(output_path), df_2025)
print(f"Saved {output_path}")
print(f"{len(expected_hours):,} hours; 263 zones; {len(df_2025):,} rows; {df_2025.trip_count.sum():,} pickups.")
print("Complete hourly coverage, pickup totals, and saved-file contents verified.")


In [ ]:
summary_2025 = pd.DataFrame(monthly_summary.values()).sort_values("month").set_index("month")
print(summary_2025.to_string())


## Features matching the three 2024 datasets

Add the same-zone lags (1, 24, 168 hours), past-only rolling means (3, 6, 24 hours), calendar features, and spatial variants. December 2024 supplies history for early January 2025. Neighbour shocks use the same frozen January–August 2024 zone/hour-of-day means as the existing 2024 evaluation features. These are one-hour-ahead features using demand observed before each forecast hour.


In [ ]:
import geopandas as gpd
import numpy as np
import pyarrow.parquet as pq

history_2024 = pd.read_parquet(
    data_dir / "yellow_taxi_2024_aggreg.parquet",
    columns = ["tpep_pickup_datetime", "PULocationID", "trip_count"],
)
year_start = pd.Timestamp("2025-01-01")
# Retain enough previous-year history for the largest lag.
december_history = history_2024.loc[
    (history_2024["tpep_pickup_datetime"] >= year_start - pd.Timedelta(hours = 168))
    & (history_2024["tpep_pickup_datetime"] < year_start)
]
feature_history = pd.concat([december_history, df_2025], ignore_index = True)
feature_history = feature_history.sort_values(["tpep_pickup_datetime", "PULocationID"]).reset_index(drop = True)

pickup_history = feature_history.pivot(
    index = "tpep_pickup_datetime", columns = "PULocationID", values = "trip_count"
).sort_index().reindex(columns = range(1, 264))
assert pickup_history.notna().all().all()
assert pickup_history.index.equals(pd.date_range(
    year_start - pd.Timedelta(hours = 168), "2026-01-01", freq = "h", inclusive = "left"
))

X_2025_basic = df_2025.copy()
row_2025 = pickup_history.index.get_indexer(X_2025_basic["tpep_pickup_datetime"])
col_2025 = pickup_history.columns.get_indexer(X_2025_basic["PULocationID"])
assert (row_2025 >= 0).all() and (col_2025 >= 0).all()
for name, hours_back in [("lag_1h", 1), ("lag_24h", 24), ("lag_1w", 168)]:
    X_2025_basic[name] = pickup_history.shift(hours_back).to_numpy()[row_2025, col_2025]
X_2025_basic["is_weekend"] = (X_2025_basic["tpep_pickup_datetime"].dt.dayofweek >= 5).astype(int)
for window in [3, 6, 24]:
    rolling = pickup_history.shift(1).rolling(window).mean()
    X_2025_basic[f"roll_mean_{window}"] = rolling.to_numpy()[row_2025, col_2025]
X_2025_basic["hours"] = X_2025_basic["tpep_pickup_datetime"].dt.hour
X_2025_basic["day_of_the_week"] = X_2025_basic["tpep_pickup_datetime"].dt.dayofweek
X_2025_basic["month"] = X_2025_basic["tpep_pickup_datetime"].dt.month
print("Created base 2025 features using December 2024 history.")


In [ ]:
# Same boundary-touching adjacency as the 2024 spatial notebook.
zones_2025 = gpd.read_file(project_dir / "taxi_zones/taxi_zones.shp")
zones_2025 = zones_2025.set_index("LocationID").reindex(range(1, 264))
assert zones_2025.geometry.notna().all()
adjacency_2025 = np.zeros((263, 263))
for i in range(263):
    shared = zones_2025.boundary.intersection(zones_2025.geometry.iloc[i].boundary)
    adjacency_2025[i] = (~shared.is_empty).to_numpy().astype(float)
np.fill_diagonal(adjacency_2025, 0)
assert np.array_equal(adjacency_2025, adjacency_2025.T)
neighbor_counts_2025 = adjacency_2025.sum(axis = 1)
has_neighbors_2025 = neighbor_counts_2025 > 0

# Select only the previous-hour observations needed for 2025 rows.
hours_2025 = pd.DatetimeIndex(sorted(df_2025["tpep_pickup_datetime"].unique()))
previous_times_2025 = hours_2025 - pd.Timedelta(hours = 1)
previous_demand_2025 = pickup_history.reindex(previous_times_2025).to_numpy()
neighbor_mean_2025 = previous_demand_2025 @ adjacency_2025.T
neighbor_mean_2025[:, has_neighbors_2025] /= neighbor_counts_2025[has_neighbors_2025]
neighbor_mean_2025[:, ~has_neighbors_2025] = 0
spatial_row_2025 = hours_2025.get_indexer(X_2025_basic["tpep_pickup_datetime"])
X_2025_meanPU = X_2025_basic.copy()
X_2025_meanPU["neighbors_mean_PU"] = neighbor_mean_2025[spatial_row_2025, col_2025]
print("Created neighbour mean at t − 1 hour.")


In [ ]:
# Reuse the 2024 training reference; do not estimate means from 2025 outcomes.
reference_2024 = history_2024.loc[history_2024["tpep_pickup_datetime"] < pd.Timestamp("2024-09-01")].copy()
reference_2024["hour_of_day"] = reference_2024["tpep_pickup_datetime"].dt.hour
zone_hour_means_2024 = (
    reference_2024.groupby(["hour_of_day", "PULocationID"])["trip_count"].mean()
    .unstack("PULocationID").reindex(index = range(24), columns = range(1, 264))
)
assert zone_hour_means_2024.notna().all().all()
previous_baseline_2025 = zone_hour_means_2024.reindex(previous_times_2025.hour).to_numpy()
previous_shocks_2025 = previous_demand_2025 - previous_baseline_2025
neighbor_shock_2025 = previous_shocks_2025 @ adjacency_2025.T
neighbor_shock_2025[:, has_neighbors_2025] /= neighbor_counts_2025[has_neighbors_2025]
neighbor_shock_2025[:, ~has_neighbors_2025] = 0
X_2025_shock = X_2025_basic.copy()
X_2025_shock["neighbors_mean_shock_1h"] = neighbor_shock_2025[spatial_row_2025, col_2025]
print("Created neighbour shocks with the frozen January–August 2024 reference.")


In [ ]:
variants_2025 = [
    (X_2025_basic, "yellow_taxi_2024_features.parquet", "yellow_taxi_2025_features.parquet"),
    (X_2025_meanPU, "yellow_taxi_2024_features_neighbor_mean.parquet", "yellow_taxi_2025_features_neighbor_mean.parquet"),
    (X_2025_shock, "yellow_taxi_2024_features_neighbor_shock.parquet", "yellow_taxi_2025_features_neighbor_shock.parquet"),
]
for dataset, template_name, filename in variants_2025:
    template_schema = pq.read_schema(data_dir / template_name)
    assert set(dataset.columns) == set(template_schema.names)
    dataset = dataset[template_schema.names]
    # Match both column order and data types to the corresponding 2024 file.
    for field in template_schema:
        dataset[field.name] = dataset[field.name].astype(field.type.to_pandas_dtype())
    pd.testing.assert_frame_equal(dataset[df_2025.columns], df_2025, check_dtype = False)
    assert len(dataset) == 365 * 24 * 263
    assert dataset.notna().all().all()
    path = data_dir / filename
    dataset.to_parquet(path, index = False)
    pd.testing.assert_frame_equal(pd.read_parquet(path), dataset)
    assert pq.read_schema(path).equals(template_schema, check_metadata = False)
    print(f"Saved {filename}: {len(dataset):,} rows, {len(dataset.columns)} columns; 2024 schema matched and read-back verified.")
